In [17]:
!apt-get update
!apt-get install ghostscript -y

!pip install "camelot-py[cv]" opencv-python-headless

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ghostscript is already the newest version (9.55.0~dfsg1-0ubuntu5.13).
0 upgraded, 0 newly inst

In [18]:
import os
import requests
import urllib.parse
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE_URL = "https://www.fia.com"

# The specific 2025 Season URL you provided
SEASON_BASE_URL = "https://www.fia.com/documents/championships/fia-formula-one-world-championship-14/season/season-2025-2071"

# The specific races you want to scrape right now
RACES = [
    "Abu Dhabi Grand Prix",
    "Australian Grand Prix",
    "Austrian Grand Prix",
    "Azerbaijan Grand Prix",
    "Bahrain Grand Prix",
    "Belgian Grand Prix",
    "British Grand Prix",
    "Canadian Grand Prix",
    "Chinese Grand Prix",
    "Dutch Grand Prix",
    "Emilia Romagna Grand Prix",
    "Hungarian Grand Prix",
    "Italian Grand Prix",
    "Japanese Grand Prix",
    "Las Vegas Grand Prix",
    "Mexico City Grand Prix",
    "Miami Grand Prix",
    "Monaco Grand Prix",
    "Qatar Grand Prix",
    "São Paulo Grand Prix",
    "Singapore Grand Prix",
    "Spanish Grand Prix",
    "United States Grand Prix"
]

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

TARGET_KEYWORDS = ["car presentation", "automobile display"]

def setup_download_folder(folder_name="fia_pdfs"):
    """Creates a local directory to save the downloaded PDFs."""
    if not os.path.exists(folder_name):
        os.makedirs(folder_name)
        print(f"Created folder: {folder_name}")
    return folder_name

def get_pdf_links(url, race_name):
    """Scrapes the FIA page and returns a list of URLs for matching PDFs."""
    print(f"\nScanning {race_name} for upgrade documents...")

    try:
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"  -> Failed to connect to the FIA website: {e}")
        return []

    soup = BeautifulSoup(response.content, 'html.parser')
    found_links = []

    for row in soup.find_all('li', class_='document-row'):
        title_element = row.find(class_='title')
        link_element = row.find('a')

        if title_element and link_element:
            text = title_element.get_text(strip=True).lower()
            href = link_element.get('href')

            if any(keyword in text for keyword in TARGET_KEYWORDS):
                full_pdf_url = urljoin(BASE_URL, href)

                safe_race_name = race_name.replace(" ", "_")
                filename = f"{safe_race_name}_{text}"

                found_links.append({"title": filename, "url": full_pdf_url})

    return found_links

def download_pdfs(pdf_list, download_folder):
    """Downloads each PDF in the list to the specified folder."""
    if not pdf_list:
        print("  -> No matching documents found on this page.")
        return

    print(f"  -> Found {len(pdf_list)} document(s)! Starting download...")

    for item in pdf_list:
        title = item['title'].replace(" ", "_").replace("/", "-") # Clean up title for filename
        pdf_url = item['url']

        if not title.endswith(".pdf"):
            title += ".pdf"

        file_path = os.path.join(download_folder, title)

        if os.path.exists(file_path):
            print(f"  -> Already exists: {title}")
            continue

        print(f"  -> Downloading: {title}...")

        try:
            doc_response = requests.get(pdf_url, headers=HEADERS, stream=True)
            doc_response.raise_for_status()

            with open(file_path, 'wb') as f:
                for chunk in doc_response.iter_content(chunk_size=8192):
                    f.write(chunk)
            print("  -> Success!")

        except Exception as e:
            print(f"  -> Failed to download {title}: {e}")

if __name__ == "__main__":
    save_dir = setup_download_folder()

    for race in RACES:
        encoded_race = urllib.parse.quote(race)

        race_url = f"{SEASON_BASE_URL}/event/{encoded_race}"

        target_pdfs = get_pdf_links(race_url, race)

        download_pdfs(target_pdfs, save_dir)

    print("\nScraping process complete!")


Scanning Abu Dhabi Grand Prix for upgrade documents...
  -> Found 1 document(s)! Starting download...
  -> Already exists: Abu_Dhabi_Grand_Prix_doc_8_-_car_presentation_submissions.pdf

Scanning Australian Grand Prix for upgrade documents...
  -> Found 1 document(s)! Starting download...
  -> Already exists: Australian_Grand_Prix_doc_7_-_car_presentation_submissions.pdf

Scanning Austrian Grand Prix for upgrade documents...
  -> Found 1 document(s)! Starting download...
  -> Already exists: Austrian_Grand_Prix_doc_9_-_car_presentation_submissions.pdf

Scanning Azerbaijan Grand Prix for upgrade documents...
  -> Found 1 document(s)! Starting download...
  -> Already exists: Azerbaijan_Grand_Prix_doc_8_-_car_presentation_submissions.pdf

Scanning Bahrain Grand Prix for upgrade documents...
  -> Found 1 document(s)! Starting download...
  -> Already exists: Bahrain_Grand_Prix_doc_10_-_car_presentation_submissions.pdf

Scanning Belgian Grand Prix for upgrade documents...
  -> Found 1 docu

In [19]:
!pip install pdfplumber

In [20]:
import os
import camelot
import pdfplumber
import pandas as pd
import re

CONSTRUCTORS = {
    "MCLAREN": "McLaren",
    "FERRARI": "Ferrari",
    "RED BULL": "Red Bull Racing",
    "MERCEDES": "Mercedes",
    "ASTON MARTIN": "Aston Martin",
    "ALPINE": "Alpine",
    "HAAS": "Haas",
    "RACING BULLS": "Visa Cash App RB",
    "WILLIAMS": "Williams",
    "SAUBER": "Kick Sauber"
}

def extract_race_name(filename):
    clean_name = filename.replace(".pdf", "")
    if "_doc_" in clean_name:
        clean_name = clean_name.split("_doc_")[0]
    if " - " in clean_name:
        clean_name = clean_name.split(" - ")[0]

    clean_name = clean_name.replace("_", " ").strip()
    return re.sub(r'^\d{4}\s+', '', clean_name)

def get_team_per_page(file_path):
    page_team_map = {}
    with pdfplumber.open(file_path) as pdf:
        for idx, page in enumerate(pdf.pages):
            page_num = idx + 1
            page_text = page.extract_text() or ""

            current_team = "Unknown Team"
            for key, formal_name in CONSTRUCTORS.items():
                if key in page_text.upper():
                    current_team = formal_name
                    break
            page_team_map[page_num] = current_team

    return page_team_map

def parse_with_camelot(pdf_folder="fia_pdfs"):
    all_rows = []

    print("Parsing PDFs using Camelot with Merged-Cell Memory...\n")

    for filename in os.listdir(pdf_folder):
        if not filename.endswith(".pdf"):
            continue

        file_path = os.path.join(pdf_folder, filename)
        race_name = extract_race_name(filename)

        print(f"Analyzing: {filename}")

        page_team_map = get_team_per_page(file_path)
        tables = camelot.read_pdf(file_path, pages='all', flavor='lattice')

        for table in tables:
            df = table.df
            if df.empty:
                continue

            team_name = page_team_map.get(table.page, "Unknown Team")

            last_valid_description = ""

            for _, row in df.iterrows():
                row_list = [str(cell).replace('\n', ' ').strip() for cell in row]

                if not any(row_list):
                    continue

                combined_text = " ".join(row_list).upper()
                if "UPDATED COMPONENT" in combined_text or "PRIMARY REASON" in combined_text or "GEOMETRIC DIFFERENCES" in combined_text:
                    continue

                if len(row_list) >= 5:
                    component = row_list[1]
                    reason = row_list[2]
                    geometry = row_list[3]
                    description = row_list[4]
                elif len(row_list) == 4:
                    component = row_list[0]
                    reason = row_list[1]
                    geometry = row_list[2]
                    description = row_list[3]
                else:
                    continue

                if description == "" and component != "":
                    description = last_valid_description
                elif description != "":
                    last_valid_description = description

                if component != "" and len(component) > 2:
                    all_rows.append({
                        "Race": race_name,
                        "Team": team_name,
                        "Component": component,
                        "Reason": reason,
                        "Geometry_Changes": geometry,
                        "Description": description
                    })

    if not all_rows:
        return pd.DataFrame()

    final_df = pd.DataFrame(all_rows, columns=["Race", "Team", "Component", "Reason", "Geometry_Changes", "Description"])

    for col in ["Component", "Reason", "Geometry_Changes", "Description"]:
        final_df[col] = final_df[col].str.replace(r'\s+', ' ', regex=True).str.strip()

    return final_df

if __name__ == "__main__":
    df_articles = parse_with_camelot("fia_pdfs")

    if not df_articles.empty:
        print("\n=== PARSING COMPLETE ===")
        print(f"Total structured rows gathered: {len(df_articles)}")
        df_articles.to_csv("f1_upgrades_structured.csv", index=False)
        print("Saved cleanly to 'f1_upgrades_structured.csv'")

        display(df_articles.head(20))
    else:
        print("\n❌ No clean table rows were successfully extracted.")

Parsing PDFs using Camelot with Merged-Cell Memory...

Analyzing: Chinese_Grand_Prix_doc_9_-_car_presentation_submissions.pdf
Analyzing: Austrian_Grand_Prix_doc_9_-_car_presentation_submissions.pdf
Analyzing: British_Grand_Prix_doc_9_-_car_presentation_submissions.pdf


Analyzing: Canadian_Grand_Prix_doc_8_-_car_presentation_submissions.pdf
Analyzing: Belgian_Grand_Prix_doc_6_-_car_presentation_submissions.pdf


Analyzing: United_States_Grand_Prix_doc_10_-_car_presentation_submissions.pdf
Analyzing: Abu_Dhabi_Grand_Prix_doc_8_-_car_presentation_submissions.pdf
Analyzing: Emilia_Romagna_Grand_Prix_doc_8_-_car_presentation_submissions.pdf


Analyzing: Qatar_Grand_Prix_doc_8_-_car_presentation_submissions.pdf
Analyzing: Las_Vegas_Grand_Prix_doc_7_-_car_presentation_submissions.pdf
Analyzing: Australian_Grand_Prix_doc_7_-_car_presentation_submissions.pdf


Analyzing: Spanish_Grand_Prix_doc_8_-_car_presentation_submissions.pdf
Analyzing: Miami_Grand_Prix_doc_8_-_car_presentation_submissions.pdf
Analyzing: Hungarian_Grand_Prix_doc_8_-_car_presentation_submissions.pdf
Analyzing: Mexico_City_Grand_Prix_doc_7_-_car_presentation_submissions.pdf


...

/usr/local/lib/python3.12/dist-packages/camelot/parsers/base.py:132: UserWarning: page-5 is image-based, camelot only works on text-based pages.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/camelot/parsers/base.py:132: UserWarning: page-10 is image-based, camelot only works on text-based pages.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/camelot/parsers/base.py:132: UserWarning: page-12 is image-based, camelot only works on text-based pages.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/camelot/parsers/base.py:132: UserWarning: page-14 is image-based, camelot only works on text-based pages.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/camelot/parsers/base.py:132: UserWarning: page-16 is image-based, camelot only works on text-based pages.
  warnings.warn(


Analyzing: Singapore_Grand_Prix_doc_10_-_car_presentation_submissions.pdf
Analyzing: Italian_Grand_Prix_doc_8_-_car_presentation_submissions.pdf


Analyzing: Monaco_Grand_Prix_doc_11_-_car_presentation_submissions.pdf



=== PARSING COMPLETE ===
Total structured rows gathered: 307
Saved cleanly to 'f1_upgrades_structured.csv'


,Race,Team,Component,Reason,Geometry_Changes,Description
0,Chinese Grand Prix,McLaren,Rear Corner,Performance - Flow Conditioning,New Rear Brake Duct Winglet,The new Rear Brake Duct Winglet improves local...
1,Chinese Grand Prix,Unknown Team,Front Wing,Circuit specific - Balance Range,Additional FW gurney options available,For circuits with higher aero balance requirem...
2,Chinese Grand Prix,Unknown Team,Beam Wing,Circuit specific - Drag Range,Two-element beam wing with greater camber & in...,Raising the trailing edge of the beam wing and...
3,Chinese Grand Prix,Williams,Rear Beam Wing,Circuit specific – Drag Level,A new main beam wing is available this weekend...,"The new beam wing options, which have a larger..."
4,Chinese Grand Prix,Kick Sauber,Coke/Engine Cover,Performance - Flow Conditioning,Changes to the engine cover design.,This test item has a potentially positive effe...
5,Austrian Grand Prix,McLaren,Front Suspension,Performance - Flow Conditioning,Updated Front Suspension,A revision to the Front Suspension fairings ai...
6,Austrian Grand Prix,McLaren,Front Corner,Performance - Flow Conditioning,Modified Front Corner,Complementing the aforementioned Front Suspens...
7,Austrian Grand Prix,McLaren,Rear Corner,Performance - Mechanical Setup,Revised Rear Corner,Alternative rear suspension geometry which req...
8,Austrian Grand Prix,Ferrari,Floor Fences,Performance - Flow Conditioning,Redistribution of fences profiles / camber,"Not event specific, this floor package feature..."
9,Austrian Grand Prix,Ferrari,Floor Body,Performance - Flow Conditioning,Reshaped boat and tunnel expansion,"Not event specific, this floor package feature..."


In [23]:
display(df_articles.head(100))

,Race,Team,Component,Reason,Geometry_Changes,Description
0,Chinese Grand Prix,McLaren,Rear Corner,Performance - Flow Conditioning,New Rear Brake Duct Winglet,The new Rear Brake Duct Winglet improves local...
1,Chinese Grand Prix,Unknown Team,Front Wing,Circuit specific - Balance Range,Additional FW gurney options available,For circuits with higher aero balance requirem...
2,Chinese Grand Prix,Unknown Team,Beam Wing,Circuit specific - Drag Range,Two-element beam wing with greater camber & in...,Raising the trailing edge of the beam wing and...
3,Chinese Grand Prix,Williams,Rear Beam Wing,Circuit specific – Drag Level,A new main beam wing is available this weekend...,"The new beam wing options, which have a larger..."
4,Chinese Grand Prix,Kick Sauber,Coke/Engine Cover,Performance - Flow Conditioning,Changes to the engine cover design.,This test item has a potentially positive effe...
...,...,...,...,...,...,...
95,Emilia Romagna Grand Prix,Alpine,Front wing,Performance – Local load,Reprofiled front wing and flap,The front wing has been redesigned to redistri...
96,Emilia Romagna Grand Prix,Alpine,Coke/Engine Cover,Performance - Flow Conditioning,Reprofiled rear bodywork panel,The rearward bodywork panel has been reprofile...
97,Emilia Romagna Grand Prix,Haas,Floor Body,Performance - Local Load,Front floor contraction shape modified,The new front floor contraction shape facilita...
98,Emilia Romagna Grand Prix,Haas,Floor Edge,Performance - Flow Conditioning,Slimmer floor edge,This new floor edge works in conjunction with ...
